# RVC — Train từng bước trong `rvc_standalone`

## Trình tự (chạy lần lượt các ô từ trên xuống)

| Thứ tự | Nội dung | Bạn làm gì |
|---------|-----------|------------|
| Chuẩn bị | Đặt file `.wav` vào thư mục con của `rvc_standalone` | Tạo ví dụ `datasets/giong_cua_toi/` và copy audio vào |
| Bước A | Khởi tạo Python / `Config` | Chạy 1 ô code |
| Bước B | Gán tên thí nghiệm + đường dẫn giống thư mục audio | Sửa `TrainingParams` cho khớp |
| Bước 1 | **Preprocess** | Tạo `logs/<tên>/0_gt_wavs/*.wav` |
| Bước 2 | **F0 + Hubert** | Tạo `2a_f0`, `2b-f0nsf`, `3_feature768` (hoặc 256 nếu v1) |
| Bước 3 | **Train** | Sinh `G_*.pth`, `D_*.pth` (lâu) |
| Bước 4 | **Index** | Sinh `added_*.index` |
| (Tùy chọn) | Xuất model nhỏ infer | `assets/weights/...pth` |

Kết quả: checkpoint trong `logs/<tên>/`, index `added_*.index` cùng thư mục; file infer nhỏ (nếu làm bước tùy chọn) trong `assets/weights/`.

## Kiểm tra trước khi train

- **Working directory** của Jupyter = thư mục **`rvc_standalone`** (phải thấy `infer/`, `configs/`).
- Đã cài: `pip install -r requirements.txt`. Nếu lỗi **fairseq**: `python -m pip install "pip>=23.2,<24.1"` rồi cài lại (xem `README_STANDALONE.txt`).
- Đã tải trọng số: `python tools/download_assets.py`.
- **`logs/mute/`**: train RVC cần bộ file nền im lặng (đưa vào `filelist`). Nếu thiếu, copy từ bản RVC đầy đủ. Ý **Bước B** sẽ báo **THIEU** nếu chưa có.

## Đặt audio huấn luyện ở đâu

1. Tạo thư mục con, ví dụ: **`datasets/giong_cua_toi/`** (trong `rvc_standalone`).
2. Copy file **`.wav`** vào đó (nói/hát, ít nền). Khuyến nghị **tổng ≥ ~10 phút**.
3. Ở **Bước B** đặt **`trainset_dir="datasets/giong_cua_toi"`** (đường dẫn tương đối so với gốc `rvc_standalone`).
4. Tránh **dấu cách** trong đường dẫn nếu được.

Ô dưới tạo thư mục mẫu; bạn copy `.wav` vào bằng tay.

In [1]:
from pathlib import Path

thu_muc_audio = Path("datasets/giong_cua_toi")
thu_muc_audio.mkdir(parents=True, exist_ok=True)
print("Đặt file .wav vào:", thu_muc_audio.resolve())

Đặt file .wav vào: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone\datasets\giong_cua_toi


## Bước A — Khởi tạo

Chạy **một lần** sau khi mở notebook (hoặc sau khi Restart kernel).

In [2]:
import logging
import os
import pathlib
import sys

STANDALONE_ROOT = pathlib.Path.cwd().resolve()
if not (STANDALONE_ROOT / "infer" / "modules" / "train" / "train.py").is_file():
    raise SystemExit(
        "cwd phải là rvc_standalone (có infer/modules/train/train.py). "
        "Mở folder rvc_standalone làm workspace hoặc cd vào đó."
    )

os.chdir(STANDALONE_ROOT)
if str(STANDALONE_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT))

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from training_pipeline.setup_env import bootstrap
from training_pipeline.params import TrainingParams
from training_pipeline import steps as train_steps

root, config = bootstrap()
assert root == STANDALONE_ROOT

print("Gốc:", STANDALONE_ROOT)
print("python:", config.python_cmd)
print("device Hubert:", config.device)
print("=== Bước A xong ===")

INFO | Found GPU NVIDIA GeForce RTX 3050 Laptop GPU
INFO | Half-precision floating-point: True, device: cuda:0


Gốc: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone
python: d:\miniconda_env\conda_envs\rvc-gpu\python.exe
device Hubert: cuda:0
=== Bước A xong ===


## Bước B — Tham số (sửa cho khớp thư mục audio)

- **`experiment_name`**: tên thư mục `logs/<tên>/`.
- **`trainset_dir`**: **đúng** thư mục chứa `.wav`.
- **`sample_rate_label`**: `40k` / `48k` / `32k` (khớp pretrained trong `assets/`).
- **`version`**: `v2` (khuyến nghị) hoặc `v1`.
- **`gpu_devices_train`**: `0` = GPU đầu tiên; train CPU được nhưng rất chậm.
- **`skip_index`**: `False` = sau train sẽ tạo `added_*.index`; `True` = bỏ bước 4 (infer vẫn được nếu không cần index).

Ô code đếm `.wav` và kiểm tra `logs/mute`. **Chạy Bước B chỉ khi đã chạy xong Bước A.**

In [3]:
from pathlib import Path

p = TrainingParams(
    experiment_name="giong_A",
    trainset_dir="datasets/giong_cua_toi",
    sample_rate_label="40k",
    version="v2",
    if_f0=True,
    num_processes=4,
    f0_method="rmvpe",
    gpu_devices_train="0",
    total_epochs=50,
    save_every_epoch=5,
    batch_size=4,
    skip_index=False,
)

ts = Path(p.trainset_dir)
if not ts.is_dir():
    raise SystemExit(f"Chưa có thư mục {ts} — tạo và copy .wav vào.")
wavs = list(ts.glob("*.wav")) + list(ts.glob("*.WAV"))
print("Số file .wav:", len(wavs))
if not wavs:
    raise SystemExit("Thư mục trainset không có .wav")

mm = train_steps.check_mute_template(STANDALONE_ROOT)
print("logs/mute:", "THIEU" if mm else "OK", mm or "")
print("=== Bước B xong — chạy Bước 1 ===")

Số file .wav: 2
logs/mute: OK 
=== Bước B xong — chạy Bước 1 ===


## Bước 1 — Preprocess

Log: `logs/<experiment>/preprocess.log`. Kết quả: `logs/<experiment>/0_gt_wavs/`.

In [4]:
train_steps.step_preprocess(STANDALONE_ROOT, config, p)
print("=== Bước 1 xong ===")

INFO | Execute: "d:\miniconda_env\conda_envs\rvc-gpu\python.exe" infer/modules/train/preprocess.py "datasets/giong_cua_toi" 40000 4 "D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A" False 3.0


datasets/giong_cua_toi 40000 4 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A False 3.0
d:\miniconda_env\conda_envs\rvc-gpu\lib\site-packages\librosa\util\files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
start preprocess
datasets/giong_cua_toi 40000 4 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A False 3.0
datasets/giong_cua_toi 40000 4 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A False 3.0
datasets/giong_cua_toi 40000 4 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\

## Bước 2 — F0 + Hubert

Log: `logs/<experiment>/extract_f0_feature.log`. Sau khi xong có `3_feature768`, `2a_f0`, `2b-f0nsf` (nếu if_f0).

In [6]:
train_steps.step_extract_f0_and_features(STANDALONE_ROOT, config, p)
print("=== Bước 2 xong ===")

INFO | Execute: "d:\miniconda_env\conda_envs\rvc-gpu\python.exe" infer/modules/train/extract/extract_f0_print.py "D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A" 4 rmvpe


d:\miniconda_env\conda_envs\rvc-gpu\lib\site-packages\pyworld\__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
infer/modules/train/extract/extract_f0_print.py D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A 4 rmvpe
d:\miniconda_env\conda_envs\rvc-gpu\lib\site-packages\pyworld\__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
todo-f0-23
f0ing,now-0,all-23,-D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion

INFO | Execute: "d:\miniconda_env\conda_envs\rvc-gpu\python.exe" infer/modules/train/extract_feature_print.py cuda:0 1 0 0 "D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A" v2 True


infer/modules/train/extract_feature_print.py cuda:0 1 0 0 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A v2 True
exp_dir: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A
load model(s) from assets/hubert/hubert_base.pt
2026-05-03 15:39:05 | INFO | fairseq.tasks.hubert_pretraining | current directory is D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone
2026-05-03 15:39:05 | INFO | fairseq.tasks.hubert_pretraining | HubertPretrainingTask Config {'_name': 'hubert_pretraining', 'data': 'metadata', 'fine_tuning': False, 'labels': ['km'], 'label_dir': 'label', 'label_rate': 50.0, 'sample_rate': 16000, 'normalize': False, 'enable_padding': False, 'max_keep_size': None, 'max_sample_size': 250000, 'min_sample_size': 32000, 'single_target': False, 'random_crop': 

## Bước 3 — Train (lâu)

Theo dõi `logs/<experiment>/train.log`. Kết quả: `G_*.pth`, `D_*.pth` trong cùng thư mục.

In [ ]:
train_steps.step_train(STANDALONE_ROOT, config, p)
print("=== Bước 3 xong ===")

INFO | Execute: "d:\miniconda_env\conda_envs\rvc-gpu\python.exe" infer/modules/train/train.py -e "giong_A" -sr 40k -f0 1 -bs 4 -g 0 -te 50 -se 5 -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth -l 1 -c 0 -sw 0 -v v2


d:\miniconda_env\conda_envs\rvc-gpu\lib\site-packages\librosa\util\files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
d:\miniconda_env\conda_envs\rvc-gpu\lib\site-packages\librosa\util\files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
INFO:giong_A:{'data': {'filter_length': 2048, 'hop_length': 400, 'max_wav_value': 32768.0, 'mel_fmax': None, 'mel_fmin': 0.0, 'n_mel_channels': 125, 'sampling_rate': 40000, 'win_length': 2048, 'training_files': './logs\\giong_A/filelist.txt'}, 'model': {'filter_ch

## Bước 4 — Index FAISS

File dùng cho infer: **`added_*.index`** (không dùng `trained_*.index`).

Nếu không cần retrieval: ở Bước B đặt `skip_index=True` và bỏ qua ô code dưới.

In [1]:
if p.skip_index:
    print("skip_index=True — bo qua")
else:
    for line in train_steps.step_train_index(STANDALONE_ROOT, config, p):
        print(line)
    print("=== Bước 4 xong ===")

NameError: name 'p' is not defined

## (Tùy chọn) Xuất `.pth` nhỏ vào `assets/weights/`

Chạy sau Bước 3. Sửa `infer_weight_name` nếu cần.

In [ ]:
p.infer_weight_name = "giong_A_infer"
print(train_steps.step_extract_small_weights(STANDALONE_ROOT, p))

## Phụ lục — Một lệnh chạy hết (cho người đã quen)

```python
# train_steps.run_all(STANDALONE_ROOT, config, p)
```

Không khuyến nghị khi mới làm quen (khó debug khi lỗi giữa chừng).